### **Content Moderation Workflow** -- **Conditional Workflow**

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)


print(f"[INFO] Model : {model.model} Loaded Successfully!")

[INFO] Model : gemini-3.6-flash Loaded Successfully!


### **Creating the ModerationState**

In [3]:
from typing import TypedDict

class ModerationPost(TypedDict):
    post_content : str
    user_reputation: str

    formatted_post : str
    content_flag: str
    result: str

### **Node 1 : Format Post Node**

In [4]:
def format_post(state: ModerationPost):
    formatted_post = f"User {state['user_reputation']} says : {state['post_content']}"
    return {
        "formatted_post" : formatted_post
    }

### **Creating an AnalyzeResponse pydantic Model**

In [5]:
from pydantic import BaseModel, Field

class AnalyzePost(BaseModel):
    flag: str = Field(description="After analyzing the post flag it out of theese three categories ['rejected','approved','review]")

analyze_model = model.with_structured_output(AnalyzePost)

### **Node 2 : Analyze Content Node**

In [6]:
def analyze_content(state: ModerationPost):
    content = state['post_content'].lower()

    res = analyze_model.invoke(f"""
    Heyy analyze the given post and flag it out of three categories : ['rejected','approved','review']\n
    Now the obvious reasons for getting 'rejected' is if the post seems to be spam, fraudulant, malicious etc.\n
    The reason for geeting 'review' tag is if the user reputation is 'new' or not a 'trusted' user.\n
    If everything seems right and the user is also trusted then flag it 'approved'.\n
    content : {content}\n
    User Reputation : {state['user_reputation']}
    """)

    flag = res.flag
    return {
        'content_flag' : flag
    }

### **Node 3 : Approve Post**

In [13]:
def approve_post(state: ModerationPost):
    result = "Post Approved and published successfully!"
    return {
        "result" : result
    }

### **Node 4 : Flag for review**

In [14]:
def flag_for_review(state: ModerationPost):
    result = "Your post has been sent to Human Moderation Queue."
    return {
        "result" : result
    }

### **Node 5 : Reject Post**

In [16]:
def reject_post(state: ModerationPost):
    result = "Post already deleted because of the policy violations!"
    return {
        "result" : result
    }